# IWI experiment · ResNet-18 on nighttime light: IWI and 512 features per cluster

This notebook trains ResNet-18 to predict the wealth (IWI) of a DHS cluster from its nighttime-light image alone, and saves the 512 values of the layer before the output (the global average pooling) for every cluster, one row per `cluster_id`. It is the PyTorch version of the Keras notebooks `27_Train_NL` (one model per fold) and `28_DeepFeature_NL` (`model.layers[-2]` for all clusters), on the clusters, images and spatial folds of 03.

* **Input:** the `NL.tif` image of 01 (224 × 224, one band): DMSP for the surveys up to 2013, VIIRS from 2014. Both sensors train one model, like the `NL_DMSP_All` models of 27_Train_NL. The image is clipped to 0–150 and normalised with `NormFunc` of the survey year, as in 04 to 07 and 28_DeepFeature_NL.
* **Target:** `iwi` of `train_clusters_folds.csv`. Clusters without an IWI value get features but are not trained on.
* **Model:** torchvision's ResNet-18 with a first convolution for one band, global average pooling to 512 features, and one linear output. Training starts from the ImageNet weights; the first-layer weights are the sum of the RGB weights, so a grey image gets the ImageNet response. `PRETRAINED = False` starts from random weights, like 27_Train_NL.
* **Training,** as in 27_Train_NL: Adam with learning rate 1e-3, MSE loss, batch 64, 100 epochs, no augmentation, and the weights with the lowest validation loss.
* **Folds:** the spatial folds of 03. Each of the five models tests on one fold, validates on the fold before it and trains on the other three:

| model | test fold | validation fold | training folds | Keras name |
|---|---|---|---|---|
| `test_E` | E | D | A, B, C | Fold12345 |
| `test_A` | A | E | B, C, D | Fold23451 |
| `test_B` | B | A | C, D, E | Fold34512 |
| `test_C` | C | B | D, E, A | Fold45123 |
| `test_D` | D | C | E, A, B | Fold51234 |

**Output,** in `nl_resnet18_runs/<sensors>_<start>/` (by default `DMSP_VIIRS_imagenet/`):

* `test_<fold>/features.csv`: the features of all sets of that model, one row for every cluster with a valid NL image, like the `NL_DMSP_All_Fold12345.csv` files of 28_DeepFeature_NL. In `test_E/features.csv`, for example, the clusters of folds A, B and C have `role` train, those of fold D val and those of fold E test. Columns: `cluster_id`, `country`, `year`, `rural`, `lat`, `lon`, `nl_sensor`, `fold`, `role`, `iwi`, `iwi_pred` and `nl_feat_0` … `nl_feat_511`. For a second-stage model tested on fold f (for example the ridge regression of 29_RidgeRegression_Landsat-NL), use the file of model `test_f`: the network that made those features never saw fold f.
* `features_oof.csv`: every cluster once, with the features and prediction of the model for which its fold was the test fold. It is written when all five models are done.
* `test_<fold>/test_predictions.csv`, `result.json`, `history.csv` and `best.pt` (the weights), and the tables and plot of section 5.

Clusters whose NL image is missing or has no-data pixels get no features; the NL cache lists them with the reason.

**Running.** Check `WORK_DIR` and `TORCH_HOME` in Settings, and download the ResNet-18 weights once (section 3; the same file as for 06 and 07). The NL caches of 04 to 07 (`table2_cache/<sensor>_NL`) are used as they are, and a missing one (usually DMSP) is built first. Every model saves its state after each epoch, so a stopped job continues where it stopped.

**From random weights.** `TRAIN_FROM_SCRATCH=1` in the environment is the same as `PRETRAINED = False`; those runs go to `nl_resnet18_runs/<sensors>_scratch/`.

> **Numbering.** This notebook was written before the notebooks were renumbered for this repository. Apart from the title, its text, comments and plot labels use the earlier numbers: 05 = `04_Table2_DHS`, 07 = `05_Table2_ResNet18`, 08 = `06_NL_ResNet18_Features`, 09 = `07_Landsat_ResNet18_Features`, 10 = `08_Landsat_ViTS16_Features`, 11 = `09_Ridge_NL_Landsat` and 12 = `10_Ridge_NL_ViT`; 01 to 03 are unchanged. The earlier 04 and 06 belonged to the simulated dataset of the preliminary study and are not included. Folder names such as `ridge_runs/nl08_landsat09` keep the earlier numbers.

# Import libraries

In [ ]:
import os
import json
import time
import uuid
import multiprocessing

# Same as in 01-07: the GPU image sets PROJ_LIB to a folder without proj.db, which breaks rasterio's CRS handling.
for _var in ('PROJ_LIB', 'PROJ_DATA'):
    if os.environ.get(_var) and not os.path.exists(os.path.join(os.environ[_var], 'proj.db')):
        os.environ.pop(_var)

import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:                      # progress bars are optional
    def tqdm(it, **kw):
        return it

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------------
WORK_DIR   = '/path/to/data/'
FOLDS_CSV  = os.path.join(WORK_DIR, 'train_clusters_folds.csv')   # written by 03_Spatial_Folds
NL_CACHE   = os.path.join(WORK_DIR, 'table2_cache')               # NL images, ~0.1 MB per cluster (shared with 04-07)
TORCH_HOME = '/path/to/torch_home'   # ResNet-18 ImageNet weights

# ---- data ---------------------------------------------------------------------------------------------
TARGET     = 'iwi'                 # International Wealth Index of the cluster (0-100), a column of FOLDS_CSV
NL_SENSORS = ['DMSP', 'VIIRS']     # 01: DMSP for the surveys up to 2013, VIIRS from 2014. Both: all clusters in one
                                   # model, like the NL_DMSP_All models of 27_Train_NL; ['VIIRS']: only 2014 and later
NODATA     = -999999.0             # no-data value of the 01 files; clusters with a no-data NL pixel get no features

# ---- normalisation of the NL image (fixed numbers, as in 04-07 and 28_DeepFeature_NL) ------------------------
# clipped to NL_CLIP, then (NL - Mean) / stdDev with NormFunc of the survey year
NL_CLIP = (0, 150)         # as in 26_tfrecords_generation_DMSP_VIIRS (VIIRS max 150, DMSP max 63); None: no clipping


def NormFunc(Year):
    if (Year >= 2014):     # VIIRS
        Mean = 0.1326654328849849
        stdDev = 1.7479698750873318
    else:                  # DMSP
        Mean = 4.152879625271373
        stdDev = 3.4762732210548317

    return Mean, stdDev


# ---- model and training, as in 27_Train_NL ------------------------------------------------------------------
FOLDS       = ['A', 'B', 'C', 'D', 'E']
EPOCHS      = 100
BATCH_SIZE  = 64
LR          = 1e-3
L2_HEAD     = 0.0          # weight penalty on the output layer: none in 27_Train_NL (04-07: 0.01, as in the paper)
SEED        = 3127
PRETRAINED  = True         # start from the ImageNet weights (False: random start, like 27_Train_NL)
AMP         = True         # bfloat16 autocast on the GPU for training; the features are computed in float32
GPU_DATA_GB = 60           # the NL images go into GPU memory when they need less than this (60 000 clusters: 6 GB),
                           # otherwise every batch is read from the caches (slower)
N_WORKERS   = len(os.sched_getaffinity(0))   # processes that build an NL cache if it is missing

# ---- TRAIN_FROM_SCRATCH=1 in the environment is the same as PRETRAINED = False:
# random start, runs in <sensors>_scratch
_scratch = os.environ.get('TRAIN_FROM_SCRATCH', '0').strip()
assert _scratch in ('', '0', '1'), f'TRAIN_FROM_SCRATCH must be 0 or 1, not {_scratch!r}'
if _scratch == '1':
    PRETRAINED = False

# ---- output -------------------------------------------------------------------------------------------------
FEATURE_PREFIX = 'nl_feat_'   # columns nl_feat_0 ... nl_feat_511 ('FeatureReleased_': the names of 28_DeepFeature_NL)
RUN_ROOT = os.path.join(WORK_DIR, 'nl_resnet18_runs',                # one folder per test fold, and the results
                        '_'.join(NL_SENSORS) + ('_imagenet' if PRETRAINED else '_scratch'))

# ---- what to run: 'all', 'none' (only the caches and the results) or test folds, comma-separated (for example A,B).
# These can also be set in the environment.
RUN_FOLDS = os.environ.get('NLFEAT_FOLDS', 'all')
RUN_FOLDS = FOLDS if RUN_FOLDS == 'all' else [] if RUN_FOLDS == 'none' else [f.strip() for f in RUN_FOLDS.split(',')]
assert set(RUN_FOLDS) <= set(FOLDS), f'unknown test folds: {RUN_FOLDS}'
assert set(NL_SENSORS) <= {'DMSP', 'VIIRS'}, NL_SENSORS
KERAS_NAME = {'E': 'Fold12345', 'A': 'Fold23451', 'B': 'Fold34512', 'C': 'Fold45123', 'D': 'Fold51234'}  # 27_Train_NL

os.environ['TORCH_HOME'] = TORCH_HOME
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(no GPU: only for code tests)')
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cache workers:', N_WORKERS)
print('sensors:', NL_SENSORS, '| ImageNet start:', PRETRAINED, '| test folds to run:', RUN_FOLDS, '| runs ->', RUN_ROOT)

# 1. Clusters

All clusters of `train_clusters_folds.csv` whose nighttime light comes from one of `NL_SENSORS`. For every sensor they are in the order of 04 to 07 (sorted by `cluster_id`), so that the NL caches are shared. They keep the fold of 03. `NormFunc` picks the DMSP or the VIIRS numbers by the survey year; the cell checks that this agrees with the sensor of every cluster.

In [ ]:
table = pd.read_csv(FOLDS_CSV)
if 'nl_sensor' not in table:                       # 01 uses DMSP up to 2013 and VIIRS from 2014
    table['nl_sensor'] = np.where(table['year'] >= 2014, 'VIIRS', 'DMSP')
if 'folder' not in table:
    table['folder'] = [os.path.join(WORK_DIR, 'Dataset', str(c).replace('/', '_')) for c in table['cluster_id']]
assert TARGET in table, f'{FOLDS_CSV} has no column {TARGET}'

# the clusters of every sensor in the order of 04-07, then one table: part = sensor, part_row = row in its NL cache
PARTS = {s: table[table['nl_sensor'] == s].sort_values('cluster_id').reset_index(drop=True) for s in NL_SENSORS}
clusters = pd.concat([p.assign(part=s, part_row=np.arange(len(p))) for s, p in PARTS.items()], ignore_index=True)
assert clusters['cluster_id'].is_unique, 'a cluster_id appears twice in FOLDS_CSV'
has_target = clusters[TARGET].notna().to_numpy()

mismatch = (clusters['nl_sensor'] == 'VIIRS') != (clusters['year'] >= 2014)
if mismatch.any():
    print(f'WARNING: {mismatch.sum()} clusters whose sensor does not match NormFunc of their year (NormFunc uses the '
          f'year):\n{clusters.loc[mismatch, ["cluster_id", "year", "nl_sensor"]].head().to_string()}\n')
left = table.loc[~table['nl_sensor'].isin(NL_SENSORS), 'nl_sensor']
print(f'{len(clusters)} clusters', clusters['nl_sensor'].value_counts().to_dict(),
      f'| {has_target.sum()} with {TARGET} | {clusters["country"].nunique()} countries, '
      f'years {clusters["year"].min()}-{clusters["year"].max()}'
      + (f' | left out (other sensor): {left.value_counts().to_dict()}' if len(left) else ''))
print(f'{TARGET}:', clusters.loc[has_target, TARGET].describe().round(2).to_dict())
if clusters['fold'].isna().any():
    print(f"{clusters['fold'].isna().sum()} clusters have no fold: they get features but no model is trained or "
          'tested on them')
print()
print(pd.crosstab(clusters['nl_sensor'], clusters['fold'], margins=True).to_string())

# 2. NL cache

The `NL.tif` image of every cluster is stored once as float16 in `table2_cache/<sensor>_NL/`, the folders and format of 04 to 07 (about 0.1 MB per cluster). A complete cache is used as it is, a missing one is built, and a job that finds a cache being built by another job (of 04 to 08) waits for it. The clipping and `NormFunc` are applied when the images are loaded for training (`NLData`, section 3). The table at the end shows the resulting training values for a sample of clusters of every sensor.

In [ ]:
PATCH = 224
NL_DIRS = {s: os.path.join(NL_CACHE, f'{s}_NL') for s in NL_SENSORS}      # the NL cache folders of 04 to 07


def cluster_nl(folder, year):
    """The NL band of one cluster (float32, shape (1, 224, 224)) and its mean, or None if it has no-data pixels."""
    with rasterio.open(os.path.join(folder, 'NL.tif')) as src:
        nl = src.read(1).astype(np.float64)
    if (nl == NODATA).any():
        return None
    return nl[None].astype(np.float32), float(nl.mean())


def _cache_one(task):
    """Read one cluster and write it at its place in the .part file. Returns the mean NL and the pixel sums."""
    i, folder, year, offset, part = task
    try:
        out = cluster_nl(folder, year)
    except Exception as e:
        return i, np.nan, None, f'{type(e).__name__}: {e}'[:200]
    if out is None:
        return i, np.nan, None, 'no-data pixels'
    x, nl_mean = out
    fd = os.open(part, os.O_WRONLY)
    try:
        os.pwrite(fd, x.astype(np.float16).tobytes(), offset)
    finally:
        os.close(fd)
    x64 = x.astype(np.float64)
    return i, nl_mean, np.stack([x64.sum(axis=(1, 2)), (x64 ** 2).sum(axis=(1, 2))]), ''


def build_cache(cache_dir, table):
    """features.npy, band_stats.csv and index.csv of the NL cache of the clusters in table (the format of 04 to 07)."""
    t0 = time.time()
    features = os.path.join(cache_dir, 'features.npy')
    shape = (len(table), 1, PATCH, PATCH)
    arr = np.lib.format.open_memmap(features + '.part', mode='w+', dtype=np.float16, shape=shape)
    header, per_cluster = arr.offset, int(np.prod(shape[1:])) * 2
    del arr
    tasks = [(i, r.folder, int(r.year), header + i * per_cluster, features + '.part')
             for i, r in enumerate(table.itertuples())]
    nl_mean = np.full(len(table), np.nan)
    sums = np.zeros((2, 1))
    error = [''] * len(table)
    with multiprocessing.get_context('fork').Pool(N_WORKERS) as pool:
        for i, t, s, err in tqdm(pool.imap_unordered(_cache_one, tasks, chunksize=4), total=len(tasks), desc='nl'):
            nl_mean[i], error[i] = t, err
            if s is not None:
                sums += s
    ok = np.isfinite(nl_mean)
    n_pix = ok.sum() * PATCH * PATCH
    mean = sums[0] / n_pix
    std = np.sqrt(np.maximum(sums[1] / n_pix - mean ** 2, 1e-12))
    pd.DataFrame({'band': ['NL'], 'mean': mean, 'std': std}).to_csv(os.path.join(cache_dir, 'band_stats.csv'),
                                                                    index=False)
    os.replace(features + '.part', features)
    index = table[['cluster_id', 'country', 'year', 'lat', 'lon']].copy()
    index['nl_mean'], index['ok'], index['error'] = nl_mean, ok, error
    index.to_csv(os.path.join(cache_dir, 'index.csv'), index=False)    # the cache counts as complete once this exists
    print(f'nl cache: {ok.sum()} of {len(ok)} clusters in {time.time() - t0:.0f} s -> {features}')


def cache_is_complete(cache_dir, table):
    index, features = os.path.join(cache_dir, 'index.csv'), os.path.join(cache_dir, 'features.npy')
    if not (os.path.exists(index) and os.path.exists(features)):
        return False
    return pd.read_csv(index)['cluster_id'].tolist() == table['cluster_id'].tolist()


def ensure_cache(cache_dir, table):
    """Build an NL cache folder unless it is complete. Waits when another job is building it (same lock in 04 to 08)."""
    os.makedirs(cache_dir, exist_ok=True)
    lock = os.path.join(cache_dir, 'building.lock')
    if cache_is_complete(cache_dir, table):
        print('NL cache found:', cache_dir)
        return
    try:
        os.mkdir(lock)
    except FileExistsError:
        print(f'another job is building {cache_dir} (if none is running, delete {lock}); waiting ...')
        t_wait = time.time()
        while os.path.exists(lock) and time.time() - t_wait < 4 * 3600:
            time.sleep(30)
        assert cache_is_complete(cache_dir, table), 'the cache is still not complete: start the notebook again'
        return
    try:
        if not cache_is_complete(cache_dir, table):     # another job may have finished it just now
            build_cache(cache_dir, table)
    finally:
        os.rmdir(lock)


def nl_norm_params(pos):
    """NormFunc mean and standard deviation (float32 arrays) of the clusters at the rows pos of `clusters`."""
    params = [NormFunc(int(y)) for y in clusters['year'].to_numpy()[pos]]
    return np.array([m for m, _ in params], np.float32), np.array([s for _, s in params], np.float32)


def read_nl(pos):
    """Cached float16 NL images, shape (len(pos), 1, 224, 224), of the clusters at the rows pos of `clusters`."""
    pos = np.asarray(pos)
    out = np.empty((len(pos), 1, PATCH, PATCH), np.float16)
    part, prow = clusters['part'].to_numpy()[pos], clusters['part_row'].to_numpy()[pos]
    for s in np.unique(part):
        k = np.flatnonzero(part == s)
        k = k[np.argsort(prow[k])]                      # read in file order
        out[k] = NLF[s][prow[k]]
    return out


def normalise_nl(x, nl_mean, nl_std):
    """Training values of NL images (float32 tensor (n, 1, 224, 224)): clipped to NL_CLIP, then NormFunc (one mean and
    standard deviation per image, from nl_norm_params)."""
    if NL_CLIP is not None:
        x = x.clamp(*NL_CLIP)
    return (x - nl_mean.view(-1, 1, 1, 1)) / nl_std.view(-1, 1, 1, 1)


for s in NL_SENSORS:
    ensure_cache(NL_DIRS[s], PARTS[s])
NL_INDEX = {s: pd.read_csv(os.path.join(NL_DIRS[s], 'index.csv')) for s in NL_SENSORS}
NLF = {s: np.load(os.path.join(NL_DIRS[s], 'features.npy'), mmap_mode='r') for s in NL_SENSORS}
for s in NL_SENSORS:
    assert NLF[s].shape == (len(PARTS[s]), 1, PATCH, PATCH), (s, NLF[s].shape)
    print(f'{s}: NL cache {NLF[s].shape}, {NLF[s].nbytes / 1e9:.1f} GB -> {NL_DIRS[s]}')
nl_ok = np.concatenate([NL_INDEX[s]['ok'].to_numpy(bool) for s in NL_SENSORS])
nl_error = pd.Series(np.concatenate([NL_INDEX[s]['error'].fillna('').to_numpy(str) for s in NL_SENSORS]))
print(f'clusters with a valid NL image: {nl_ok.sum()} of {len(nl_ok)}; left out:',
      nl_error[~nl_ok].str.split(':').str[0].value_counts().to_dict())

# training values of the NL image for a sample of clusters of every sensor
values = []
for s in NL_SENSORS:
    cand = np.flatnonzero(nl_ok & (clusters['part'] == s).to_numpy())
    if not len(cand):
        continue
    sample = np.sort(np.random.default_rng(0).choice(cand, size=min(64, len(cand)), replace=False))
    m, sd = nl_norm_params(sample)
    raw = torch.from_numpy(read_nl(sample).astype(np.float32))
    x = normalise_nl(raw, torch.from_numpy(m), torch.from_numpy(sd))
    values.append({'sensor': s, 'clusters': len(sample),
                   'NormFunc (Mean, stdDev)': ', '.join(f'({a:.3f}, {b:.3f})' for a, b in sorted(set(zip(m, sd)))),
                   'cached min': raw.min().item(), 'cached mean': raw.mean().item(), 'cached max': raw.max().item(),
                   'training min': x.min().item(), 'training mean': x.mean().item(), 'training max': x.max().item()})
print('\nNL image for a sample of clusters, as cached and as the network sees it (clipped, NormFunc):')
pd.DataFrame(values).round(3)

In [ ]:
# check one cluster of every sensor against its file, then show the training input of clusters with low, middle and
# high IWI
usable = nl_ok & has_target
fig, ax = plt.subplots(len(NL_SENSORS), 3, figsize=(12, 4.2 * len(NL_SENSORS)), constrained_layout=True, squeeze=False)
for r, s in enumerate(NL_SENSORS):
    cand = np.flatnonzero(usable & (clusters['part'] == s).to_numpy())
    if not len(cand):
        for a in ax[r]:
            a.axis('off')
        continue
    order = cand[np.argsort(clusters[TARGET].to_numpy()[cand], kind='stable')]
    picks = np.array([order[0], order[len(order) // 2], order[-1]])           # lowest, middle and highest IWI
    i = picks[-1]
    nl_new, mean_new = cluster_nl(clusters.at[i, 'folder'], clusters.at[i, 'year'])
    assert np.allclose(read_nl([i])[0].astype(np.float32), nl_new, rtol=2e-3, atol=2e-3)
    assert np.isclose(mean_new, NL_INDEX[s].at[clusters.at[i, 'part_row'], 'nl_mean'])
    print(f'{s}: cache matches the file for', clusters.at[i, 'cluster_id'])
    m, sd = nl_norm_params(picks)
    x = normalise_nl(torch.from_numpy(read_nl(picks).astype(np.float32)), torch.from_numpy(m), torch.from_numpy(sd))
    for c, (i, img) in enumerate(zip(picks, x[:, 0].numpy())):
        a = ax[r, c]
        fig.colorbar(a.imshow(img, cmap='magma'), ax=a, shrink=0.8)
        a.set_title(f"{clusters.at[i, 'cluster_id']} ({s}, {clusters.at[i, 'year']})\n"
                    f"{TARGET} {clusters.at[i, TARGET]:.1f}", fontsize=9)
        a.set_xticks([])
        a.set_yticks([])
fig.suptitle('Training input: NL clipped and normalised with NormFunc (clusters with low, middle and high IWI)')
plt.show()

# 3. Model and training

`NLResNet18` is torchvision's ResNet-18 for one input band. `features()` gives the 512 values of the global average pooling, the input of the output layer, and `head`, a linear layer that runs in float32, turns them into the IWI. It corresponds to `build_resnet18((224, 224, 1), 1)` of 27_Train_NL (which also has a 1 × 1 shortcut convolution in the first stage), whose `model.layers[-2]` is the same global average pooling. The output layer has no weight penalty (`L2_HEAD = 0`), as in 27_Train_NL, and its bias starts at the mean IWI of the training clusters. `NLData` keeps the NL images of all clusters in GPU memory and returns normalised batches.

A run trains the model of one test fold. After every epoch it writes the history, the best weights (`best.pt`, lowest validation loss) and its full state (`last.pt`). Then it loads `best.pt`, computes the features and the prediction of every cluster in float32 and writes `features.csv`, `test_predictions.csv` and `result.json`, and deletes `last.pt`, which is only needed to continue a stopped run. A run with `result.json` and `features.csv` is skipped. A run with only a `last.pt` continues from that epoch. A finished run whose `features.csv` was deleted writes it again from `best.pt`, for example after changing `FEATURE_PREFIX`.

If loading the ImageNet weights fails because the machine has no internet access, download them once on a machine that has it, then run again:

```bash
mkdir -p /path/to/torch_home/hub/checkpoints
wget -P /path/to/torch_home/hub/checkpoints \
     https://download.pytorch.org/models/resnet18-f37072fd.pth
```

In [ ]:
RESNET_STATE = None
if PRETRAINED:
    try:
        RESNET_STATE = torchvision.models.resnet18(weights='IMAGENET1K_V1').state_dict()
    except Exception as e:
        raise RuntimeError(f'ResNet-18 ImageNet weights could not be loaded ({type(e).__name__}: {e}). '
                           'Download them as shown above.') from e


class NLResNet18(nn.Module):
    """torchvision ResNet-18 for one input band. features(): the 512 values of the global average pooling (the input
    of the output layer, model.layers[-2] in 28_DeepFeature_NL); forward(): the predicted IWI."""

    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet18(weights=None)
        if RESNET_STATE is not None:
            net.load_state_dict(RESNET_STATE)
        rgb_weights = net.conv1.weight.detach().clone()           # (64, 3, 7, 7)
        net.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            if RESNET_STATE is not None:     # sum of the RGB weights (their mean x 3, as in 06/07): a grey image gets
                net.conv1.weight.copy_(rgb_weights.sum(dim=1, keepdim=True))       # the ImageNet response
            else:                            # torchvision's initialisation of its convolutions
                nn.init.kaiming_normal_(net.conv1.weight, mode='fan_out', nonlinearity='relu')
        net.fc = nn.Identity()                                     # the network now ends with the 512 pooled values
        self.backbone = net
        self.head = nn.Linear(512, 1)
        nn.init.xavier_uniform_(self.head.weight)                  # Keras defaults: Glorot uniform, zero bias
        nn.init.zeros_(self.head.bias)

    def features(self, x):
        return self.backbone(x)

    def forward(self, x):
        f = self.features(x)
        with torch.autocast(device_type=f.device.type, enabled=False):    # output layer in float32
            return self.head(f.float()).squeeze(1)


class NLData:
    """The normalised NL images of the clusters at the rows `rows` of `clusters` (clipped to NL_CLIP, NormFunc of
    the survey year). The cached values go into DEVICE memory when they take less than GPU_DATA_GB, otherwise every
    batch is read from the caches."""

    def __init__(self, rows):
        self.rows = np.asarray(rows)
        m, sd = nl_norm_params(self.rows)
        self.nl_mean = torch.from_numpy(m).to(DEVICE)          # NormFunc of every cluster, by position in rows
        self.nl_std = torch.from_numpy(sd).to(DEVICE)
        gb = len(self.rows) * PATCH * PATCH * 2 / 1e9
        self.x = None
        if gb < GPU_DATA_GB:
            self.x = torch.empty((len(self.rows), 1, PATCH, PATCH), dtype=torch.float16, device=DEVICE)
            for s in tqdm(range(0, len(self.rows), 1024), desc=f'loading NL images, {gb:.1f} GB', leave=False):
                self.x[s:s + 1024] = torch.from_numpy(read_nl(self.rows[s:s + 1024])).to(DEVICE)

    def batch(self, idx):
        """Normalised float32 images of the clusters at positions idx (a 1-D tensor)."""
        if self.x is not None:
            x = self.x[idx.to(self.x.device)]
        else:
            x = torch.from_numpy(read_nl(self.rows[idx.cpu().numpy()])).to(DEVICE, non_blocking=True)
        idx = idx.to(DEVICE)
        x = normalise_nl(x.float(), self.nl_mean[idx], self.nl_std[idx])
        return x.contiguous(memory_format=torch.channels_last)


def r2_score(y, p):
    y, p = np.asarray(y, np.float64), np.asarray(p, np.float64)
    return float(1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16, enabled=AMP and DEVICE.type == 'cuda')


def predict(model, data, idx, batch=256):
    model.eval()
    out = []
    with torch.no_grad(), autocast():
        for s in range(0, len(idx), batch):
            out.append(model(data.batch(idx[s:s + batch])).float())
    return torch.cat(out)


def save(obj, path):
    torch.save(obj, path + '.tmp')
    os.replace(path + '.tmp', path)


def extract(model, data, idx, batch=256):
    """Features (n, 512) and predictions (n,) of the clusters at positions idx, in float32 (no autocast)."""
    model.eval()
    feats, preds = [], []
    with torch.no_grad():
        for s in range(0, len(idx), batch):
            f = model.features(data.batch(idx[s:s + batch])).float()
            feats.append(f.cpu())
            preds.append(model.head(f).squeeze(1).cpu())
    return torch.cat(feats).numpy(), torch.cat(preds).numpy()


def write_csv(frame, path, **kw):
    """Write a CSV file under a temporary name, then rename it: nobody (another run) reads half a file."""
    tmp = f'{path}.{uuid.uuid4().hex[:8]}.tmp'
    frame.to_csv(tmp, index=False, **kw)
    os.replace(tmp, path)

In [ ]:
META = ['cluster_id', 'country', 'year', 'rural', 'lat', 'lon', 'nl_sensor', 'fold']    # columns of features.csv


def run_fold(data, y, folds, test_fold):
    """Train (or continue) the model of one test fold, then write the features and the prediction of every cluster
    (features.csv), the test predictions and result.json. y: IWI at the positions of data.rows (NaN: none); folds:
    their folds of 03 ('' : none). Returns the test metrics (also in result.json)."""
    run_dir = os.path.join(RUN_ROOT, f'test_{test_fold}')
    result_file, features_file = os.path.join(run_dir, 'result.json'), os.path.join(run_dir, 'features.csv')
    if os.path.exists(result_file) and os.path.exists(features_file):
        with open(result_file) as f:
            return json.load(f)
    os.makedirs(run_dir, exist_ok=True)
    k = FOLDS.index(test_fold)
    val_fold = FOLDS[(k - 1) % len(FOLDS)]
    train_folds = [f for f in FOLDS if f not in (test_fold, val_fold)]
    has_y = np.isfinite(y)
    pos = lambda mask: torch.from_numpy(np.flatnonzero(mask)).to(DEVICE)
    tr = pos(has_y & np.isin(folds, train_folds))
    va, te = pos(has_y & (folds == val_fold)), pos(has_y & (folds == test_fold))
    y_dev = torch.tensor(np.nan_to_num(y), dtype=torch.float32, device=DEVICE)

    torch.manual_seed(SEED + k)
    model = NLResNet18().to(DEVICE).to(memory_format=torch.channels_last)
    with torch.no_grad():
        model.head.bias.fill_(float(y_dev[tr].mean()))
    opt = torch.optim.Adam(model.parameters(), lr=LR, eps=1e-7)    # Keras epsilon
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(SEED * 100 + k)
    start, best, best_epoch = 0, float('inf'), -1
    last_file, best_file, hist_file = (os.path.join(run_dir, f) for f in ('last.pt', 'best.pt', 'history.csv'))
    if os.path.exists(result_file):                     # finished before: only features.csv is written again
        assert os.path.exists(best_file), f'{best_file} is missing: delete {result_file} to train this model again'
        start = EPOCHS
    elif os.path.exists(last_file):                     # continue a run that was stopped
        st = torch.load(last_file, map_location=DEVICE)
        model.load_state_dict(st['model'])
        opt.load_state_dict(st['opt'])
        gen.set_state(st['gen'].cpu())
        start, best, best_epoch = st['epoch'] + 1, st['best'], st['best_epoch']
        if os.path.exists(hist_file):                   # epochs after the saved state are trained again
            hist = pd.read_csv(hist_file)
            hist[hist['epoch'] <= start].to_csv(hist_file, index=False)
        print(f'test {test_fold}: continuing at epoch {start + 1}')
    elif os.path.exists(hist_file):
        os.remove(hist_file)                            # history of a run that stopped before saving its state

    for epoch in range(start, EPOCHS):
        t0 = time.time()
        model.train()
        perm = tr[torch.randperm(len(tr), generator=gen, device=DEVICE)]
        sq_sum = torch.zeros((), device=DEVICE)
        n_seen = 0
        for s in range(0, len(perm) - BATCH_SIZE + 1, BATCH_SIZE):     # last incomplete batch dropped, as in 27
            b = perm[s:s + BATCH_SIZE]
            with autocast():
                pred = model(data.batch(b))
            mse = F.mse_loss(pred.float(), y_dev[b])
            loss = mse + L2_HEAD * model.head.weight.pow(2).sum()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sq_sum += mse.detach() * len(b)
            n_seen += len(b)
        reg = L2_HEAD * float(model.head.weight.detach().pow(2).sum())
        p_va = predict(model, data, va).cpu().numpy()
        y_va = y[va.cpu().numpy()]
        val_mse = float(((p_va - y_va) ** 2).mean())
        val_loss = val_mse + reg
        if val_loss < best:
            best, best_epoch = val_loss, epoch
            save({'model': model.state_dict(), 'epoch': epoch}, best_file)
        rec = {'epoch': epoch + 1, 'train_mse': float(sq_sum) / max(n_seen, 1), 'l2_term': reg, 'val_mse': val_mse,
               'val_loss': val_loss, 'val_r2': r2_score(y_va, p_va), 'seconds': round(time.time() - t0, 1)}
        pd.DataFrame([rec]).to_csv(hist_file, mode='a', header=not os.path.exists(hist_file), index=False)
        save({'model': model.state_dict(), 'opt': opt.state_dict(), 'gen': gen.get_state(), 'epoch': epoch,
              'best': best, 'best_epoch': best_epoch}, last_file)
        if epoch == start or (epoch + 1) % 10 == 0 or epoch + 1 == EPOCHS:
            print(f"test {test_fold} epoch {epoch + 1}/{EPOCHS}: train mse {rec['train_mse']:.2f}, val mse "
                  f"{val_mse:.2f}, val R2 {rec['val_r2']:.3f}, best epoch {best_epoch + 1}, {rec['seconds']} s")

    # the best weights: features and prediction of every cluster, test metrics
    st = torch.load(best_file, map_location=DEVICE)
    model.load_state_dict(st['model'])
    feats, pred = extract(model, data, torch.arange(len(data.rows), device=DEVICE))
    role = np.select([np.isin(folds, train_folds), folds == val_fold, folds == test_fold], ['train', 'val', 'test'], '')
    n_set = {r: int((role == r).sum()) for r in ('train', 'val', 'test', '')}       # rows of features.csv per set
    out = pd.concat([clusters.iloc[data.rows][[c for c in META if c in clusters]].reset_index(drop=True),
                     pd.DataFrame({'role': role, TARGET: y, f'{TARGET}_pred': pred}),
                     pd.DataFrame(feats, columns=[f'{FEATURE_PREFIX}{j}' for j in range(feats.shape[1])])], axis=1)
    write_csv(out.sort_values('cluster_id'), features_file, float_format='%.7g')

    te_np, va_np = te.cpu().numpy(), va.cpu().numpy()
    p_te, y_te = pred[te_np], y[te_np]
    test_rows = clusters.iloc[data.rows[te_np]]
    write_csv(test_rows[['cluster_id', 'country', 'year', 'nl_sensor']].assign(y_true=y_te, y_pred=p_te),
              os.path.join(run_dir, 'test_predictions.csv'))
    result = {'test_fold': test_fold, 'val_fold': val_fold, 'train_folds': ','.join(train_folds),
              'keras_name': KERAS_NAME[test_fold], 'sensors': ','.join(NL_SENSORS), 'pretrained': PRETRAINED,
              'n_train': len(tr), 'n_val': len(va), 'n_test': len(te), 'n_features': len(out),
              'best_epoch': st['epoch'] + 1, 'epochs': EPOCHS, 'val_r2': r2_score(y[va_np], pred[va_np]),
              'test_r2': r2_score(y_te, p_te), 'test_rmse': float(np.sqrt(((p_te - y_te) ** 2).mean())),
              'test_mae': float(np.abs(p_te - y_te).mean()), 'features_train': n_set['train'],
              'features_val': n_set['val'], 'features_test': n_set['test']}
    with open(result_file + '.tmp', 'w') as f:
        json.dump(result, f, indent=1)
    os.replace(result_file + '.tmp', result_file)
    if os.path.exists(last_file):
        os.remove(last_file)                            # only needed to continue a stopped run
    print(f"test {test_fold} ({KERAS_NAME[test_fold]}): test R2 {result['test_r2']:.3f}, RMSE {result['test_rmse']:.2f}"
          f" (best epoch {result['best_epoch']})\n    features of all sets -> {features_file}: train {n_set['train']} "
          f"({', '.join(train_folds)}), val {n_set['val']} ({val_fold}), test {n_set['test']} ({test_fold})"
          + (f", no fold {n_set['']}" if n_set[''] else ''))
    return result

# 4. Train the five models

The models train on the clusters with a valid NL image, an IWI value and a fold of 03, and compute the features of all clusters with a valid NL image. The images are loaded once for all test folds in `RUN_FOLDS`.

In [ ]:
rows = np.flatnonzero(nl_ok)                                   # rows of `clusters` that get features
y = clusters[TARGET].to_numpy(dtype=np.float64)[rows]          # NaN: no IWI, not trained on
folds = clusters['fold'].fillna('').astype(str).to_numpy()[rows]
trainable = np.isfinite(y) & np.isin(folds, FOLDS)
print(f'{len(rows)} clusters with a valid NL image, {trainable.sum()} of them with {TARGET} and a fold:',
      pd.Series(folds[trainable]).value_counts().sort_index().to_dict())
todo = [f for f in RUN_FOLDS
        if not all(os.path.exists(os.path.join(RUN_ROOT, f'test_{f}', n)) for n in ('result.json', 'features.csv'))]
if todo and DEVICE.type != 'cuda':
    print('WARNING: no GPU, training will be very slow')
if todo:
    data = NLData(rows)
    for f in todo:
        print(f'\n=== test fold {f} ({KERAS_NAME[f]}), validation fold {FOLDS[(FOLDS.index(f) - 1) % len(FOLDS)]}')
        run_fold(data, y, folds, f)
    del data
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
else:
    print(f'test folds {RUN_FOLDS}: already done' if RUN_FOLDS else 'NLFEAT_FOLDS=none: no training, only the results')

# 5. Results and the features of every cluster

For every model: the test R², RMSE and MAE (in IWI points), the validation R² of the best weights, and the number of clusters of each set in its `features.csv` (`features_train`, `features_val`, `features_test`; they include the clusters without IWI). Below: the mean over the models, and the R² of all test predictions pooled, also for each sensor. The tables go to `nl_resnet18_per_fold.csv` and `nl_resnet18_results.csv` in `RUN_ROOT`. When all five models are done, the test rows of their `features.csv` files are put together in `features_oof.csv`.

Reading the features, for example of model `test_E` (Keras Fold12345):

```python
feat = pd.read_csv(os.path.join(RUN_ROOT, 'test_E', 'features.csv'))
X = feat.filter(like=FEATURE_PREFIX).to_numpy()          # (clusters, 512)
train, test = feat['role'].isin(['train', 'val']), feat['role'] == 'test'
```

In [ ]:
def scores(p):
    err = p['y_pred'] - p['y_true']
    return {'clusters': len(p), 'R2': r2_score(p['y_true'], p['y_pred']) if len(p) > 1 else np.nan,
            'RMSE': float(np.sqrt((err ** 2).mean())), 'MAE': float(err.abs().mean())}


found, preds = [], []
for f in FOLDS:
    run_dir = os.path.join(RUN_ROOT, f'test_{f}')
    if os.path.exists(os.path.join(run_dir, 'result.json')):
        with open(os.path.join(run_dir, 'result.json')) as fh:
            found.append(json.load(fh))
        preds.append(pd.read_csv(os.path.join(run_dir, 'test_predictions.csv')))
COLS = ['test_fold', 'val_fold', 'train_folds', 'keras_name', 'n_train', 'n_val', 'n_test', 'best_epoch', 'val_r2',
        'test_r2', 'test_rmse', 'test_mae', 'features_train', 'features_val', 'features_test']
per_fold = pd.DataFrame(found, columns=COLS + ['n_features', 'sensors', 'pretrained', 'epochs'])
pooled = pd.concat(preds, ignore_index=True) if preds else None
summary = []
if len(per_fold):
    summary.append({'what': f'mean of {len(per_fold)} model' + 's' * (len(per_fold) > 1),
                    'clusters': int(per_fold['n_test'].sum()),
                    'R2': per_fold['test_r2'].mean(), 'R2 sd': per_fold['test_r2'].std(),
                    'RMSE': per_fold['test_rmse'].mean(), 'MAE': per_fold['test_mae'].mean()})
    summary.append({'what': 'pooled test predictions', **scores(pooled)})
    for s, p in pooled.groupby('nl_sensor'):
        summary.append({'what': f'pooled, {s}', **scores(p)})
summary = pd.DataFrame(summary, columns=['what', 'clusters', 'R2', 'R2 sd', 'RMSE', 'MAE'])
os.makedirs(RUN_ROOT, exist_ok=True)
write_csv(per_fold, os.path.join(RUN_ROOT, 'nl_resnet18_per_fold.csv'))
write_csv(summary, os.path.join(RUN_ROOT, 'nl_resnet18_results.csv'))

# features_oof.csv: every cluster with the features of the model for which its fold was the test fold
oof_file = os.path.join(RUN_ROOT, 'features_oof.csv')
feature_files = [os.path.join(RUN_ROOT, f'test_{f}', 'features.csv') for f in FOLDS]
missing = [f for f, p in zip(FOLDS, feature_files) if not os.path.exists(p)]
if missing:
    print('features_oof.csv is written when all five models are done; missing test folds:', missing)
elif not os.path.exists(oof_file) or os.path.getmtime(oof_file) < max(map(os.path.getmtime, feature_files)):
    oof = pd.concat([pd.read_csv(p).query("role == 'test'") for p in feature_files], ignore_index=True)
    assert oof['cluster_id'].is_unique
    write_csv(oof.sort_values('cluster_id'), oof_file, float_format='%.7g')
    print(f'features of {len(oof)} clusters, each from the model that did not see its fold -> {oof_file}')
else:
    print('features_oof.csv is up to date:', oof_file)

print(f'\nResNet-18 on {" + ".join(NL_SENSORS)} nighttime light ({"ImageNet" if PRETRAINED else "random"} start), '
      f'{TARGET} target, spatial folds of 03, {len(per_fold)} of {len(FOLDS)} models done')
display(per_fold[COLS].round(3))
summary.round(3)

In [ ]:
if pooled is not None and len(pooled) > 1:
    blues = LinearSegmentedColormap.from_list('blues', ['#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#184f95',
                                                        '#0d366b'])          # one hue, light (few) to dark (many)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 5.4), constrained_layout=True,
                                 gridspec_kw={'width_ratios': [1.2, 1]})
    lo = float(min(pooled['y_true'].min(), pooled['y_pred'].min()))
    hi = float(max(pooled['y_true'].max(), pooled['y_pred'].max()))
    lo, hi = lo - 0.03 * (hi - lo), hi + 0.03 * (hi - lo)
    hb = a1.hexbin(pooled['y_true'], pooled['y_pred'], gridsize=40, extent=(lo, hi, lo, hi), mincnt=1, cmap=blues,
                   linewidths=0.2, edgecolors='face')
    fig.colorbar(hb, ax=a1, shrink=0.85, label='clusters', ticks=MaxNLocator(integer=True))
    a1.plot([lo, hi], [lo, hi], color='#52514e', linewidth=1, linestyle='--')
    a1.text(0.98, 0.02, '1:1', transform=a1.transAxes, ha='right', va='bottom', fontsize=8, color='#52514e')
    a1.set_xlim(lo, hi)
    a1.set_ylim(lo, hi)
    a1.set_aspect('equal')
    a1.set_xlabel(f'observed {TARGET.upper()}')
    a1.set_ylabel(f'predicted {TARGET.upper()}')
    a1.text(0.03, 0.97, '\n'.join(f"{r.what}: R² {r.R2:.2f}, RMSE {r.RMSE:.1f} (n = {r.clusters})"
                                  for r in summary.itertuples() if r.what.startswith('pooled')),
            transform=a1.transAxes, va='top', fontsize=8, color='#0b0b0b')
    a1.set_title('Test predictions of the models, pooled', loc='left', fontsize=11)

    yy = np.arange(len(per_fold))[::-1]
    a2.barh(yy, per_fold['test_r2'], height=0.6, color='#2a78d6')
    for k, v in zip(yy, per_fold['test_r2']):
        if v > 0.15:                                       # value inside the bar
            a2.text(0.01, k, f'{v:.2f}', va='center', ha='left', color='white', fontsize=8)
        else:
            a2.text(max(v, 0) + 0.015, k, f'{v:.2f}', va='center', ha='left', color='#52514e', fontsize=8)
    r2_pooled = summary.loc[summary['what'] == 'pooled test predictions', 'R2'].iloc[0]
    a2.axvline(r2_pooled, color='#52514e', linewidth=1, linestyle='--')
    a2.text(r2_pooled, len(per_fold) - 0.45, f' pooled {r2_pooled:.2f}', ha='left', va='bottom', fontsize=8,
            color='#52514e')
    a2.set_yticks(yy)
    a2.set_yticklabels([f'test {r.test_fold} ({r.keras_name})' for r in per_fold.itertuples()])
    a2.set_ylim(-0.6, len(per_fold) - 0.1)
    a2.set_xlim(min(0, per_fold['test_r2'].min() - 0.05), 1)
    a2.set_xlabel('test R²')
    a2.grid(axis='x', color='#e1e0d9', linewidth=0.8)
    a2.set_axisbelow(True)
    a2.set_title('Test R² of each model', loc='left', fontsize=11)
    for a in (a1, a2):
        for side in ('top', 'right'):
            a.spines[side].set_visible(False)
    fig.suptitle(f'ResNet-18 on nighttime light ({" + ".join(NL_SENSORS)}, '
                 f'{"ImageNet" if PRETRAINED else "random"} start), {TARGET.upper()}, spatial 5-fold CV', fontsize=12)
    png = os.path.join(RUN_ROOT, 'nl_resnet18_predictions.png')
    fig.savefig(png + '.tmp.png', dpi=150)
    os.replace(png + '.tmp.png', png)
    plt.show()
else:
    print('no finished models yet')